In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os

model = SentenceTransformer('all-MiniLM-L6-v2')

# Path to data.txt (file crated by me)
data_path = os.path.expanduser("~/Desktop/data.txt")

with open(data_path, "r") as f:
    texts = f.readlines()

# Converting text to embeddings
embeddings = model.encode(texts)

# Creating the FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

# Save index and texts to Desktop
faiss.write_index(index, os.path.expanduser("~/Desktop/index.faiss"))
np.save(os.path.expanduser("~/Desktop/texts.npy"), texts)

print("✅ Embeddings created successfully!")

# Save
faiss.write_index(index, "index.faiss")
np.save("texts.npy", np.array(texts))  # Convert texts to numpy array before saving

print("Embeddings created!")

Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 4933.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings created successfully!
Embeddings created!


In [8]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import os

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Load saved index and texts from Desktop
index = faiss.read_index(os.path.expanduser("~/Desktop/index.faiss"))
texts = np.load(os.path.expanduser("~/Desktop/texts.npy"), allow_pickle=True)

def ask(question, k=5, keyword=None):
    """
    Retrieve top k similar texts from FAISS index.
    Optionally filter results containing the keyword.
    """
    q_embedding = model.encode([question])
    D, I = index.search(q_embedding, k=k)
    results = [texts[i] for i in I[0]]

    # Filter by keyword if provided - fixed indentation here
    if keyword:
        results = [r for r in results if keyword.lower() in r.lower()]

    return results  # fixed indentation here - return must be inside the function

if __name__ == "__main__":
    # Example question
    query = "When should I change the engine oil in my car?"
    answers = ask(query, k=5, keyword="engine oil") # filter for relevant sentences

    print("🔍 Question:", query)
    print("📄 Retrieved Results:")
    if answers:
        for ans in answers:
            print("-", ans.strip())
    else:
        print("No relevant results found.")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 10075.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 Question: When should I change the engine oil in my car?
📄 Retrieved Results:
- Engine oil should be changed every 10,000 km.
